# 🐍 Clase 8 · Construir MatchingEngine

> Programar el algoritmo que convierte Order + OrderBook en fills y estado nuevo, reutilizando un único proceso para MARKET, LIMIT, IOC y FOK.

**Hoy construyes:** MatchingEngine: planificar cruces, validar y mutar el libro.

⏱️ 🟢 núcleo ~24 min · 🔵 si vamos bien +49 min.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Algunos ejercicios traen una **💭 Pista** intermedia; si no basta, abre **💡 Ver solución**.

Cada ejercicio lleva su etiqueta: **🟢 núcleo** (en clase) · **🔵 si vamos bien** · **🟣 bonus** (el cuaderno de auxiliares profundiza más).

### B1 · ¿Qué lado consumo?

<sub>🟢 núcleo · ~4 min</sub>

Implementa `opposite_levels`: BUY devuelve asks; SELL devuelve bids. Devuelve la lista real del book, no una copia.

<sub>practicas: selección BUY/SELL</sub>

In [ ]:
from exchange.book import OrderBook, Level
from exchange.orders import Order, OrderType, Side
book = OrderBook('BTC', [Level(100,2),Level(99,3)], [Level(101,1),Level(102,4)])
def opposite_levels(order, book):
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert opposite_levels.__code__.co_consts != (None,), '⏸ implementa opposite_levels: su cuerpo sigue siendo pass'
buy=Order('BTC',Side.BUY,1,order_type=OrderType.MARKET)
sell=Order('BTC',Side.SELL,1,order_type=OrderType.MARKET)
assert opposite_levels(buy,book) is book.asks
assert opposite_levels(sell,book) is book.bids
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
def opposite_levels(order, book):
    return book.asks if order.side is Side.BUY else book.bids
```

</details>

### B2 · remaining + take

<sub>🟢 núcleo · ~6 min</sub>

Implementa `plan_market`. Devuelve `(planned, remaining)` y no cambies el libro. Cada plan es `(price, take)`.

<sub>practicas: planificar sin mutar</sub>

In [ ]:
from exchange.book import OrderBook, Level
from exchange.orders import Order, OrderType
book=OrderBook('BTC',[Level(100,2)],[Level(101,.8),Level(102,1),Level(103,2)])
order=Order('BTC','buy',2.4,order_type=OrderType.MARKET)
def plan_market(order, book):
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert plan_market.__code__.co_consts != (None,), '⏸ implementa plan_market: su cuerpo sigue siendo pass'
before=[(x.price,x.size) for x in book.asks]
planned,remaining=plan_market(order,book)
assert [(p,round(s,9)) for p,s in planned]==[(101,.8),(102,1),(103,.6)]
assert abs(remaining)<1e-12
assert [(x.price,x.size) for x in book.asks]==before, 'PLAN no muta'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
def plan_market(order, book):
    remaining=order.size; planned=[]
    opposite=book.asks if order.side.value=='buy' else book.bids
    for level in opposite:
        if remaining<=1e-12: break
        take=min(remaining,level.size)
        planned.append((level.price,take)); remaining-=take
    return planned,remaining
```

</details>

### B3 · Primera MARKET completa

<sub>🟢 núcleo · ~9 min</sub>

Implementa `StudentMatchingEngine.process` para MARKET. Debe barrer varios niveles, crear fills y mutar correctamente el book.

<sub>practicas: loop → reduce → Fill</sub>

In [ ]:
from exchange.book import OrderBook, Level
from exchange.orders import Order, OrderType, Side
from exchange.trades import Fill
class StudentMatchingEngine:
    def process(self, order, book, timestamp=None):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMatchingEngine.process.__code__.co_consts != (None,), '⏸ implementa StudentMatchingEngine.process: su cuerpo sigue siendo pass'
book=OrderBook('BTC',[Level(100,2)],[Level(101,.8),Level(102,1),Level(103,2)])
order=Order('BTC','buy',2.4,order_type=OrderType.MARKET)
fills=StudentMatchingEngine().process(order,book,7)
assert [(f.price,round(f.size,9)) for f in fills]==[(101,.8),(102,1),(103,.6)]
assert [(x.price,round(x.size,9)) for x in book.asks]==[(103,1.4)]
assert all(f.timestamp==7 and f.side is Side.BUY for f in fills)
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    def process(self, order, book, timestamp=None):
        opposite=book.asks if order.side is Side.BUY else book.bids
        remaining=order.size; planned=[]
        for level in opposite:
            if remaining<=1e-12: break
            take=min(remaining,level.size); planned.append((level.price,take)); remaining-=take
        fills=[]
        for price,take in planned:
            book.reduce(order.side.opposite,price,take)
            fills.append(Fill(order.id,order.symbol,order.side,price,take,timestamp))
        return fills
```

</details>

### B4 · Diseña _crosses()

<sub>🟢 núcleo · ~5 min</sub>

Implementa `_crosses(order, level_price)`. BUY cruza si el nivel está a su límite o mejor; SELL, simétricamente.

<sub>practicas: LIMIT simétrica BUY/SELL</sub>

In [ ]:
from exchange.orders import Order, OrderType
def crosses(order, level_price):
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert crosses.__code__.co_consts != (None,), '⏸ implementa crosses: su cuerpo sigue siendo pass'
buy=Order('BTC','buy',1,price=101,order_type=OrderType.LIMIT)
sell=Order('BTC','sell',1,price=99,order_type=OrderType.LIMIT)
assert crosses(buy,100) and crosses(buy,101) and not crosses(buy,102)
assert crosses(sell,100) and crosses(sell,99) and not crosses(sell,98)
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
def crosses(order, level_price):
    if order.side.value=='buy': return order.price>=level_price
    return order.price<=level_price
```

</details>

### B5 · LIMIT marketable

<sub>🔵 si vamos bien · ~8 min</sub>

Añade `_crosses()` al plan. La LIMIT puede barrer precios válidos, pero debe parar en el primer nivel que empeora su límite.

<sub>practicas: detener el plan en el límite</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
class StudentMatchingEngine:
    @staticmethod
    def _crosses(order, price):
        pass
    def process(self, order, book, timestamp=None):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMatchingEngine._crosses.__code__.co_consts != (None,), '⏸ implementa StudentMatchingEngine._crosses: su cuerpo sigue siendo pass'
assert StudentMatchingEngine.process.__code__.co_consts != (None,), '⏸ implementa StudentMatchingEngine.process: su cuerpo sigue siendo pass'
b=OrderBook('BTC',[Level(99,2)],[Level(101,1),Level(102,1),Level(103,1)])
o=Order('BTC','buy',3,price=102,order_type=OrderType.LIMIT)
f=StudentMatchingEngine().process(o,b)
assert [(x.price,x.size) for x in f]==[(101,1),(102,1)]
assert [(x.price,x.size) for x in b.asks]==[(103,1)]
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    @staticmethod
    def _crosses(order, price):
        return order.price>=price if order.side is Side.BUY else order.price<=price
    def process(self, order, book, timestamp=None):
        opposite=book.asks if order.side is Side.BUY else book.bids
        remaining=order.size; planned=[]
        for level in opposite:
            if remaining<=1e-12 or not self._crosses(order,level.price): break
            take=min(remaining,level.size); planned.append((level.price,take)); remaining-=take
        fills=[]
        for price,take in planned:
            book.reduce(order.side.opposite,price,take)
            fills.append(Fill(order.id,order.symbol,order.side,price,take,timestamp))
        return fills
```

</details>

### B6 · El remanente LIMIT descansa

<sub>🔵 si vamos bien · ~7 min</sub>

Extiende LIMIT: tras el commit, cualquier `remaining` debe descansar al precio límite en el lado de la orden.

<sub>practicas: book.add_limit</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
class StudentMatchingEngine:
    # integra plan, commit y tratamiento del remanente LIMIT
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert '__init__' in vars(StudentMatchingEngine), '⏸ StudentMatchingEngine está vacía: escribe su __init__ y sus métodos'
b=OrderBook('BTC',[Level(99,2)],[Level(101,1),Level(102,1)])
o=Order('BTC','buy',3,price=101,order_type=OrderType.LIMIT)
f=StudentMatchingEngine().process(o,b)
assert sum(x.size for x in f)==1
assert any(x.price==101 and abs(x.size-2)<1e-12 for x in b.bids), 'el remanente descansa'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    @staticmethod
    def _crosses(o,p): return o.price>=p if o.side is Side.BUY else o.price<=p
    def process(self,o,b,timestamp=None):
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; plan=[]
        for lv in xs:
            if r<=1e-12 or not self._crosses(o,lv.price): break
            take=min(r,lv.size); plan.append((lv.price,take)); r-=take
        fills=[]
        for p,s in plan:
            b.reduce(o.side.opposite,p,s); fills.append(Fill(o.id,o.symbol,o.side,p,s,timestamp))
        if r>1e-12: b.add_limit(o.side,o.price,r)
        return fills
```

</details>

### B7 · Diseña IOC sin duplicar el engine

<sub>🔵 si vamos bien · ~7 min</sub>

Soporta LIMIT e IOC con un solo proceso. Ambas cruzan igual; solo LIMIT puede ejecutar `add_limit` para el remanente.

<sub>practicas: misma ejecución, otra política de remanente</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
class StudentMatchingEngine:
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert '__init__' in vars(StudentMatchingEngine), '⏸ StudentMatchingEngine está vacía: escribe su __init__ y sus métodos'
def run(kind):
 b=OrderBook('BTC',[Level(99,2)],[Level(101,1)]); o=Order('BTC','buy',3,price=101,order_type=kind); f=StudentMatchingEngine().process(o,b); return f,b
fl,bl=run(OrderType.LIMIT); fi,bi=run(OrderType.IOC)
assert [(x.price,x.size) for x in fl]==[(x.price,x.size) for x in fi]==[(101,1)]
assert any(x.price==101 and x.size==2 for x in bl.bids)
assert all(x.price!=101 for x in bi.bids), 'IOC cancela el remanente'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    @staticmethod
    def _crosses(o,p): return o.price>=p if o.side is Side.BUY else o.price<=p
    def process(self,o,b,timestamp=None):
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; plan=[]
        for lv in xs:
            if r<=1e-12 or not self._crosses(o,lv.price): break
            take=min(r,lv.size); plan.append((lv.price,take)); r-=take
        fills=[]
        for p,s in plan:
            b.reduce(o.side.opposite,p,s); fills.append(Fill(o.id,o.symbol,o.side,p,s,timestamp))
        if r>1e-12 and o.order_type is OrderType.LIMIT: b.add_limit(o.side,o.price,r)
        return fills
```

</details>

### B8 · Depura FOK: plan → validate → commit

<sub>🔵 si vamos bien · ~9 min</sub>

La implementación inicial muta mientras recorre. Repárala: una FOK imposible devuelve `[]` y deja el book nivel por nivel idéntico.

<sub>practicas: atomicidad del estado</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
class StudentMatchingEngine:
    def process(self,o,b,timestamp=None):
        # BUG: reduce antes de saber si habrá liquidez suficiente
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; fills=[]
        for lv in list(xs):
            if r<=1e-12: break
            take=min(r,lv.size); b.reduce(o.side.opposite,lv.price,take)
            fills.append(Fill(o.id,o.symbol,o.side,lv.price,take,timestamp)); r-=take
        return [] if r>1e-12 else fills

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
b=OrderBook('BTC',[Level(99,2)],[Level(101,2),Level(102,1)])
before=([(x.price,x.size) for x in b.bids],[(x.price,x.size) for x in b.asks])
o=Order('BTC','buy',10,price=102,order_type=OrderType.FOK)
assert StudentMatchingEngine().process(o,b)==[]
after=([(x.price,x.size) for x in b.bids],[(x.price,x.size) for x in b.asks])
assert after==before, 'FOK fallida debe ser atómica'
print('ok — no hubo mutación parcial')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Primero guarda `(price, take)` en una lista y calcula cuánto podrías llenar. Si no alcanza, retorna antes del primer `reduce`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    def process(self,o,b,timestamp=None):
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; plan=[]
        for lv in xs:
            if r<=1e-12: break
            if o.price is not None and not (o.price>=lv.price if o.side is Side.BUY else o.price<=lv.price): break
            take=min(r,lv.size); plan.append((lv.price,take)); r-=take
        if r>1e-12: return []
        fills=[]
        for p,s in plan:
            b.reduce(o.side.opposite,p,s); fills.append(Fill(o.id,o.symbol,o.side,p,s,timestamp))
        return fills
```

</details>

### B9 · Refactor: un único process()

<sub>🔵 si vamos bien · ~10 min</sub>

Integra los cuatro tipos en `StudentMatchingEngine`. No escribas cuatro loops: selección, plan y commit deben ser comunes.

<sub>practicas: MARKET/LIMIT/IOC/FOK reutilizando fases</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
EPS=1e-12
class StudentMatchingEngine:
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert '__init__' in vars(StudentMatchingEngine), '⏸ StudentMatchingEngine está vacía: escribe su __init__ y sus métodos'
def snap(b): return ([(x.price,round(x.size,9)) for x in b.bids],[(x.price,round(x.size,9)) for x in b.asks])
def fresh(): return OrderBook('BTC',[Level(100,1),Level(99,2)],[Level(101,1),Level(102,2)])
e=StudentMatchingEngine()
b=fresh(); assert sum(x.size for x in e.process(Order('BTC','buy',2,order_type=OrderType.MARKET),b))==2
b=fresh(); e.process(Order('BTC','buy',2,price=101,order_type=OrderType.LIMIT),b); assert (101,1) in snap(b)[0]
b=fresh(); e.process(Order('BTC','buy',2,price=101,order_type=OrderType.IOC),b); assert (101,1) not in snap(b)[0]
b=fresh(); before=snap(b); assert e.process(Order('BTC','buy',9,price=102,order_type=OrderType.FOK),b)==[] and snap(b)==before
b=fresh(); f=e.process(Order('BTC','sell',1.5,order_type=OrderType.MARKET),b); assert [x.price for x in f]==[100,99]
print('ok — cuatro políticas, un algoritmo')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    @staticmethod
    def _crosses(o,p): return o.price>=p if o.side is Side.BUY else o.price<=p
    def process(self,o,b,timestamp=None):
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; plan=[]
        for lv in xs:
            if r<=EPS: break
            if o.order_type is not OrderType.MARKET and not self._crosses(o,lv.price): break
            take=min(r,lv.size); plan.append((lv.price,take)); r-=take
        if o.order_type is OrderType.FOK and r>EPS: return []
        fills=[]
        for p,s in plan:
            b.reduce(o.side.opposite,p,s); fills.append(Fill(o.id,o.symbol,o.side,p,s,timestamp))
        if r>EPS and o.order_type is OrderType.LIMIT: b.add_limit(o.side,o.price,r)
        return fills
```

</details>

### B10 · Prueba contra la referencia

<sub>🔵 si vamos bien · ~8 min</sub>

Ejecuta escenarios simétricos contra tu engine y el de referencia. Compara fills por comportamiento y el estado final completo.

<sub>practicas: differential testing</sub>

In [ ]:
from exchange.book import OrderBook,Level
from exchange.orders import Order,OrderType,Side
from exchange.trades import Fill
from exchange.matching import MatchingEngine
EPS=1e-12
# reutiliza aquí tu StudentMatchingEngine de B9
matched = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert matched is not None, '⏸ matched sigue en None: completa el ejercicio antes de validar'
def fresh(): return OrderBook('BTC',[Level(100,1),Level(99,2)],[Level(101,1),Level(102,2)])
def state(b): return ([(x.price,round(x.size,9)) for x in b.bids],[(x.price,round(x.size,9)) for x in b.asks])
def sig(fs): return [(f.side.value,f.price,round(f.size,9)) for f in fs]
assert matched is True
for side,kind,size,px in [('buy',OrderType.MARKET,2,None),('sell',OrderType.MARKET,1.5,None),('buy',OrderType.LIMIT,2,101),('buy',OrderType.IOC,2,101),('buy',OrderType.FOK,9,102)]:
 a,c=fresh(),fresh(); oa=Order('BTC',side,size,price=px,order_type=kind); oc=Order('BTC',side,size,price=px,order_type=kind)
 fa=StudentMatchingEngine().process(oa,a); fc=MatchingEngine().process(oc,c)
 assert sig(fa)==sig(fc) and state(a)==state(c)
print('ok — coincide con la referencia')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMatchingEngine:
    @staticmethod
    def _crosses(o,p): return o.price>=p if o.side is Side.BUY else o.price<=p
    def process(self,o,b,timestamp=None):
        xs=b.asks if o.side is Side.BUY else b.bids; r=o.size; plan=[]
        for lv in xs:
            if r<=EPS: break
            if o.order_type is not OrderType.MARKET and not self._crosses(o,lv.price): break
            take=min(r,lv.size); plan.append((lv.price,take)); r-=take
        if o.order_type is OrderType.FOK and r>EPS: return []
        fills=[]
        for p,s in plan:
            b.reduce(o.side.opposite,p,s); fills.append(Fill(o.id,o.symbol,o.side,p,s,timestamp))
        if r>EPS and o.order_type is OrderType.LIMIT: b.add_limit(o.side,o.price,r)
        return fills
matched=True
```

</details>

## Cierre

Cuando una operación puede abortarse, primero planifica y valida; después muta el estado.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`matching_demo.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python matching_demo.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python matching_demo.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.